In [ ]:
# I am importing the necessary tools for the decision trees and random forest 
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree 
from sklearn.metrics import cohen_kappa_score, accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import pandas as pd
# There is no limit for how many columns and rows can be displayed 
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

In [ ]:
# I am reading in my cleaned train data, saving it as Train, and printing the first 10 rows
Train = pd.read_csv('cleaneddftrain.csv')

In [ ]:
# I am printing all of the information for Train
Train.info()

In [ ]:
# I am printing the sum for nulls for each column in Train
Train.isna().sum()

In [ ]:
#I am reading in my cleaned test data as Test and printing the sum of nulls for each column
Test = pd.read_csv("cleaneddftest.csv")
Test.isna().sum()

In [ ]:
#I am making a copy of Train and Test before I encode
Train2 = Train.copy()

In [ ]:
# This function one-hot encodes the categorical columns   
def encode(Train2): 
    # It loops through all the columns and sees which ones are objects and which ones are categories  
    for col in Train2.columns:                       
        if Train2[col].dtype == 'object' or Train2[col].dtype == 'category':  
            #Then it encodes each column that meets the criteria 
            le = LabelEncoder()                  
            Train2[col] = le.fit_transform(Train2[col]) 
    return Train2                                   
#The function was called and saved back to the original Train dataframe
Train2 = encode(Train2)     

In [ ]:
# I made a copy of Train 
Train2_clean = Train2.copy()
# For all the columns in Train_clean I looked for the ones that do not have clean in their name and dropped them 
for col in Train2_clean.columns:
    if 'clean' not in col:
        Train2_clean=Train2_clean.drop(columns=col)
#I checked to see if only clean columns remained
Train2_clean.info()
#For X_train I dropped decision clean, ID clean, SAT R Evidence-Based Reading and Writing Section + Math Section clean, train-test clean,School 1 Class Rank (Numeric) clean, School 1 Class Size (Numeric) clean 
X_train = Train2_clean.drop(
    ["Decision clean",
     "ID clean",
     "SAT R Evidence-Based Reading and Writing Section + Math Section clean",
     "train-test clean",
    "School 1 Class Rank (Numeric) clean",
    "School 1 Class Size (Numeric) clean"], axis = 1)
#For y_train I only kept Decision clean because it is what we are predicting 
y_train = Train2_clean['Decision clean']

## Simple Decision Tree with Validation 

In [ ]:
#I am splitting the Train_clean dataset into 20% validation and 80% train 
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0)

In [ ]:
# The train data is being used to fit the simple decision tree and the splits will be decided with gini impurity 
#There is a random state of 0
dtc = DecisionTreeClassifier(criterion='gini', random_state = 0)
dtc.fit(X_train,y_train)


# The y's of the validation and test sets are being predicted using the training model 
y_train_pred = dtc.predict(X_train)
y_val_pred = dtc.predict(X_val)

#The accuracy, precision, recall, f1, and cohen kappa scores for the train set are calculated 
print("sp_train_Accuracy:", accuracy_score(y_train, y_train_pred))
print("sp_train_Precision:", precision_score(y_train, y_train_pred))
print("sp_train_Recall:", recall_score(y_train, y_train_pred))
print("sp_train_F1-score:", f1_score(y_train, y_train_pred))
print("Cohen's Kappa Score_train:", cohen_kappa_score(y_train, y_train_pred))

#The accuracy, precision, recall, f1, and cohen kappa scores for the validation set are calculated 
print("\nsp_val_Accuracy:", accuracy_score(y_val, y_val_pred))
print("sp_val_Precision:", precision_score(y_val, y_val_pred))
print("sp_val_Recall:", recall_score(y_val, y_val_pred))
print("sp_val_F1-score:", f1_score(y_val, y_val_pred))
print("Cohen's Kappa Score_test:", cohen_kappa_score(y_val, y_val_pred))

In [ ]:
#Shows the number of predicted class labels in the validation set 
values, counts = np.unique(y_val_pred, return_counts=True)

print(dict(zip(values, counts)))

In [ ]:
#Shows the feature importance of each column
#Numbers are in the order of the columns in the dataset 
feature_importances = dtc.feature_importances_
feature_importances

In [ ]:
# Shows the feature importance of each column in order of how important they are 
feature_importance_dict = {'feature_name': X_train.columns, 'feature_importances': feature_importances}
feature_importance_df = pd.DataFrame(feature_importance_dict).sort_values(by='feature_importances', ascending=False)
print(feature_importance_df)

## Pruned Decision Tree with Validation

In [ ]:
#Shows how big the train and validation data sets 
print("Dataset Sizes:\n")

print(f"Training set (used to fit models): {X_train.shape[0]} observations")
print(f"Validation set (used to select ccp_alpha): {X_val.shape[0]} observations")

#Shows how many features are in the train and validation data sets 
print("\nFeature dimensions (number of columns):")
print(f"Training set: {X_train.shape[1]} features")
print(f"Validation set: {X_val.shape[1]} features")


In [ ]:
#This code is making a simple decision tree
simple_tree = DecisionTreeClassifier(criterion='gini', #It uses gini to decide how to split 
                                     max_features=X_train.shape[1], #It can use all the features within the dataset
                                     random_state=0, #The randomness has been set to 0
                                     min_samples_split=2, # Two observations are needed for a node to split 
                                     min_samples_leaf=1, #One observation is needed for a leaf node 
                                     max_depth=None, #There is no max depth
                                     ccp_alpha=0) # Should not be pruned since it is set to 0

# Looks for the best ccp_alpha values when pruning the tree
#When the tree is pruned, a new ccp_alpha value is assigned to it
path = simple_tree.cost_complexity_pruning_path(X_train, y_train)                                               
alphas = path['ccp_alphas']
impurities = path['impurities']

In [ ]:
#Stores results of pruned trees made in the loop
pruning_results = []


#Looks through all the ccp_alphas and uses that to train many different trees 
for alpha, impurity in zip(alphas, impurities):
    pruned_tree = DecisionTreeClassifier(criterion='gini', #It uses gini to decide how to split 
                                         max_features=X_train.shape[1], #It can use all the features within the dataset
                                         random_state=0,#The randomness has been set to 0
                                         min_samples_split=2,# Two observations are needed for a node to split 
                                         min_samples_leaf=1,#One observation is needed for a leaf node 
                                         max_depth=None, #There is no max depth
                                         ccp_alpha=alpha) #set to the alpha we defined in the last cell
    pruned_tree.fit(X_train, y_train)
#Takes note of the ccp_alpa, impurity, number of leaves, and depth of each pruned tree
    pruning_results.append({
        'ccp_alpha': alpha,
        'total_impurity': impurity,
        'n_leaves': pruned_tree.get_n_leaves(),
        'max_depth': pruned_tree.get_depth()
    })

# Makes the results into a dataframe 
pruning_results_df = pd.DataFrame(pruning_results)
#prints part of the dataframe 
pd.set_option('display.max_rows', 20)
pruning_results_df.head(10)

In [ ]:
# Two lists were made for each dataset to save the Cohen's kappa scores from the loop
ACC_Train_simple = []
ACC_Validation_simple = []
# loop over the alpha scores and find the Cohen's kappa score for each of the sets 
for i in alphas:
    simple_tree = DecisionTreeClassifier(criterion='gini',#It uses gini to decide how to split 
                                         max_features=X_train.shape[1],#It can use all the features within the dataset
                                         random_state=0,#The randomness has been set to 0
                                         min_samples_split=2,# Two observations are needed for a node to split 
                                         min_samples_leaf=1,#One observation is needed for a leaf node 
                                         max_depth=None,#There is no max depth
                                         ccp_alpha=i)# This is what the tree is iterating over 
    #The tree is being fit to the train2 data
    simple_tree.fit(X_train, y_train)
    #The y's are being predicted using the using the simple tree
    y_train_sub_pred = simple_tree.predict(X_train)
    y_val_sub_pred = simple_tree.predict(X_val)
    #The cohen kappa scores from the loop for both train and validation are being added to the list 
    ACC_Train_simple.append(cohen_kappa_score(y_train, y_train_sub_pred))
    ACC_Validation_simple.append(cohen_kappa_score(y_val, y_val_sub_pred))

In [ ]:
#finds the best validation score for the validation set and sees what alpha score corresponds to that
max_acc_alpha_val = alphas[ACC_Validation_simple.index(max(ACC_Validation_simple))]
max_acc_alpha_val

In [ ]:
#Makes a pruned decision tree based on the best Cohen's kappa alpha from the test set
pruned_tree = DecisionTreeClassifier(criterion='gini', #It uses gini to decide how to split 
                                     min_samples_split=2, #It can use all the features within the dataset
                                     min_samples_leaf=1, #The randomness has been set to 0
                                     max_depth=None,# Two observations are needed for a node to split 
                                     max_features=X_train.shape[1],#One observation is needed for a leaf node 
                                     random_state=0,#There is no max depth
                                     ccp_alpha=max_acc_alpha_val)# Should not be pruned since it is set to 0
pruned_tree.fit(X_train, y_train) #The pruned tree was fit to the training set 

# y values for the train, validation, and test sets are predicted using the trained pruned tree
y_train_pred_pt = pruned_tree.predict(X_train)
y_val_pred_pt = pruned_tree.predict(X_val)

In [ ]:
#The accuracy, precision, recall, f1, and cohen kappa scores for the train set are calculated 
print("pt_train_Accuracy:", accuracy_score(y_train, y_train_pred_pt))
print("pt_train_Precision:", precision_score(y_train, y_train_pred_pt))
print("pt_train_Recall:", recall_score(y_train, y_train_pred_pt))
print("pt_train_F1-score:", f1_score(y_train, y_train_pred_pt))
print("Cohen's Kappa Score_train:", cohen_kappa_score(y_train, y_train_pred_pt))

#The accuracy, precision, recall, f1, and cohen kappa scores for the validation set are calculated 
print("\npt_val_Accuracy:", accuracy_score(y_val, y_val_pred_pt))
print("pt_val_Precision:", precision_score(y_val, y_val_pred_pt))
print("pt_val_Recall:", recall_score(y_val, y_val_pred_pt))
print("pt_val_F1-score:", f1_score(y_val, y_val_pred_pt))
print("Cohen's Kappa Score_test:", cohen_kappa_score(y_val, y_val_pred_pt))

In [ ]:
#The feature importance scores for each column are calculated 
pt_feature_importances = pruned_tree.feature_importances_  

#A dictionary with the feature importance scores is made with the corresponding column names 
feature_importance_dict = {
    'feature_name': X_train.columns,  
    'pt_feature_importances': pt_feature_importances  
}
#The dictionary is made into a dataframe and sorted by highest fetaure importance to lowest 
feature_importance_df = pd.DataFrame(feature_importance_dict).sort_values(by='pt_feature_importances', ascending=False)  # Sort by importance

print(feature_importance_df)  

## Random Forest with Validation

In [ ]:
#The sizes of each of the datasets are printed 
print("Training set shape:", X_train.shape)
print("Validation set shape:", X_val.shape)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

#A Random forest is ebing made with these criteria 
Tree_RF = RandomForestClassifier(
    n_estimators=200, #There should be at least 200 trees in the forest 
    criterion='gini', #Gini is used to split 
    max_features=6, #There can not be more than 6 features per split 
    bootstrap=True, # gets the trees training sample with replacement
    max_samples=X_train.shape[0], #can get a sample size that is the same size as the training set 
    max_depth=None, #There is no max depth 
    oob_score=True, #calculates OOB accuracy 
    random_state=0 #randomness is set to 0 
)
#The random forest is fit using the train data 
Tree_RF.fit(X_train, y_train)

# Calculates the OOB accuracy which is an internal performance score that is calculated using the oob_score
print("Initial OOB Accuracy:", round(Tree_RF.oob_score_, 3))

In [ ]:
#empty lists for the validation kappa and max features 
max_features = []
val_kappa_rf = []
#This loop finds the best amount of max features for the random forest using these criteria
for i in range(1, X_train.shape[1] + 1):
    Tree_RF = RandomForestClassifier(
        n_estimators=100, #There should be at least 100 trees in the forest              
        criterion='gini',  #Gini is used to split            
        max_features=i,  #max fetaures is what the loop is iterating over       
        bootstrap=True,   # gets the trees training sample with replacement             
        max_samples=X_train.shape[0], #can get a sample size that is the same size as the training set 
        max_depth=None,  #There is no max depth               
        oob_score=True,   #calculates OOB accuracy               
        random_state=0     #randomness is set to 0              
    )
    #The random forest is being fit to the train2 data 
    Tree_RF.fit(X_train, y_train)
    #The yal_val is being predicted using the random forest
    y_val_pred = Tree_RF.predict(X_val)
    #The max features have been added to the list 
    max_features.append(i)
    #The validation Cohen's kappa scores have been added to the list 
    val_kappa_rf.append(cohen_kappa_score(y_val, y_val_pred))
    

# A data frame of the max features and their corresponding Cohen's kappa is made and printed 
#It is sorted from highest to lowest 
df_kappa_val_rf = pd.DataFrame({
    'max_features': max_features,
    'validation_kappa_rf': val_kappa_rf
}).sort_values(by='validation_kappa_rf', ascending=False)

print(df_kappa_val_rf.round(3))
# The best feature (the first one) is stored in best_max_val_features 
best_max_val_features = int(df_kappa_val_rf.iloc[0]['max_features'])
print("Optimal max_features based on validation kappa:", best_max_val_features)


In [ ]:
#The best max feature is used to make another random forest with the same criteria as before 
Tree_RF_Best = RandomForestClassifier(
    n_estimators=100,   #There should be at least 100 trees in the forest                  
    criterion='gini', #Gini is used to split                    
    max_features=best_max_val_features,   #set to the best max val features we found in the last cell
    bootstrap=True,   # gets the trees training sample with replacement                   
    max_samples=X_train.shape[0],   #can get a sample size that is the same size as the training set       
    max_depth=None,  #There is no max depth                     
    oob_score=True,   #calculates OOB accuracy                 
    random_state=0   #randomness is set to 0                     
)
#The random forest is fit using the train2 data
Tree_RF_Best.fit(X_train, y_train)

# the y's are predicted using the fitted model 
y_train_pred_RF = Tree_RF_Best.predict(X_train)
y_val_pred_RF = Tree_RF_Best.predict(X_val)

# Shows the oob prediction summary from the random forest 
oob_accuracy_RF = Tree_RF_Best.oob_score_

In [ ]:
#This function gives you the accuracy, precision, recall, f1, and cohen kappa score for both sets
def report_metrics(y_true, y_pred, label):
    print(label + "_Accuracy:", round(accuracy_score(y_true, y_pred), 3))
    print(label + "_Precision:", round(precision_score(y_true, y_pred), 3))
    print(label + "_Recall:", round(recall_score(y_true, y_pred), 3))
    print(label + "_F1_score:", round(f1_score(y_true, y_pred), 3))
    print(label + "_Kappa:", round(cohen_kappa_score(y_true, y_pred), 3))
    print()
#The performance metrics for train and validation are printed 
report_metrics(y_train, y_train_pred_RF, "RF_train")
report_metrics(y_val, y_val_pred_RF, "RF_validation")
#The OOB accuracy is printed and compared to the validation  accuracy 
print("RF_OOB_Accuracy:", round(oob_accuracy_RF, 3))
print("Validation vs OOB Accuracy Difference:", round(accuracy_score(y_val, y_val_pred_RF) - oob_accuracy_RF, 3))




In [ ]:
#The feature importance scores for each column is calculated with the best random forest model  
feature_importances = Tree_RF_Best.feature_importances_

#A dicitoonary with the column names and each columns feature importance score is made 
feature_importance_dict = {
    'feature_name': X_train.columns,
    'feature_importances': feature_importances
}
#That dictionary is then converted into a dataframe and is printed in order from highest feature importance to lowest 
feature_importance_df = pd.DataFrame(feature_importance_dict).sort_values(
    by='feature_importances', ascending=False
)
pd.set_option('display.max_rows', None)
print(feature_importance_df.round(3))

## Data Refinement Method 1: Recategorizing School 1 GPA Recalculated

In [ ]:
# I made a copy of Train
Train3 = Train.copy()

I believe this may help improve my metrics because it is kind of high on the feature importance list. It also contains many numbers so maybe recategorizing it into excellent GPA, good GPA, and bad GPA will help.

In [ ]:
#I checked value counts for School 1 GPA Recalculated clean
Train3["School 1 GPA Recalculated clean"].value_counts()
#I made a list of GPAs I believed were excellent 
excellent = [4.00, 3.99, 3.98, 3.97, 3.96, 3.95, 3.94, 3.93, 3.92, 3.91, 3.9, 3.89, 3.88, 3.87, 3.86, 
             3.85, 3.84, 3.83, 3.82, 3.81, 3.8, 3.79, 3.78, 3.77, 3.76]
#I made a list of GPAs I believed were good 
good = [3.75, 3.74, 3.73, 3.72, 3.71, 3.7, 3.69, 3.68, 3.67, 3.66, 3.65, 3.64, 3.63, 3.62, 3.61, 3.6, 
        3.59, 3.58,3.57, 3.56, 3.55, 3.54, 3.53, 3.52, 3.51, 3.5, 3.49, 3.48, 3.47, 3.46, 3.45, 3.44, 
        3.43, 3.42, 3.41, 3.4, 3.39, 3.38, 3.37, 3.36, 3.35, 3.34, 3.33, 3.32, 3.31, 3.3, 3.29, 3.28,
        3.27, 3.26, 3.25, 3.24, 3.23, 3.22, 3.21, 3.2, 3.19, 3.18, 3.17, 3.16, 3.15, 3.14, 3.13, 3.12,
        3.11, 3.1, 3.09, 3.08, 3.07, 3.06, 3.05, 3.04, 3.03, 3.02, 3.01, 3.00]
#I made a list of GPAs I believed were bad
bad = [2.99, 2.98, 2.97, 2.96, 2.95, 2.94, 2.93, 2.92, 2.91, 2.9, 2.89, 2.88, 2.87, 2.86, 2.85, 2.84, 
       2.83, 2.82, 2.81, 2.8, 2.79, 2.78, 2.77, 2.76, 2.75, 2.74, 2.73, 2.72, 2.71, 2.7, 2.69, 2.68,
       2.67, 2.66, 2.65, 2.64, 2.63, 2.62, 2.61, 2.6, 2.59, 2.58, 2.57, 2.56, 2.55, 2.54, 2.53, 2.52, 
       2.51, 2.5, 2.33]
#I recategorized School 1 GPA Recalculated clean into excellent, goodm and bad gpas 
Train3["School 1 GPA Recalculated clean"] = Train3["School 1 GPA Recalculated clean"].apply(lambda x: 'Good' if x in good else x)
Train3["School 1 GPA Recalculated clean"] = Train3["School 1 GPA Recalculated clean"].apply(lambda x: 'Excellent' if x in excellent else x)
Train3["School 1 GPA Recalculated clean"] = Train3["School 1 GPA Recalculated clean"].apply(lambda x: 'Bad' if x in bad else x)
#I printed value counts to make sure it worked 
print(Train3["School 1 GPA Recalculated clean"].value_counts())
#I label encoded it 
le = LabelEncoder()
Train3["School 1 GPA Recalculated clean"] = le.fit_transform(Train3["School 1 GPA Recalculated clean"])
#I printed value counts again to make sure the label encoding worked
print(Train3["School 1 GPA Recalculated clean"].value_counts())

In [ ]:
#This function looks through all of the columns on Train3 and encodes the categorical ones 
def encode(Train3): 
    # It loops through all the columns and sees which ones are objects and which ones are categories  
    for col in Train3.columns:                       
        if Train3[col].dtype == 'object' or Train3[col].dtype == 'category':  
            #Then it encodes each column that meets the criteria 
            le = LabelEncoder()                  
            Train3[col] = le.fit_transform(Train3[col]) 
    return Train3                                   
#The function was called and saved back to the original Train dataframe
Train3 = encode(Train3)  

In [ ]:
#I made a clean copy of Train3 to modify
Train3_clean = Train3.copy()
#This for loop looks through all of the columns in Train3 and drops the ones that do not have clean in their name 
for col in Train3_clean.columns:
    if 'clean' not in col:
        Train3_clean=Train3_clean.drop(columns=col)
#For X_train2, I dropped the columns that we do not need for the decision trees and the one that we are predicitng 
X_train2 = Train3_clean.drop(
    ["Decision clean",
     "ID clean",
     "SAT R Evidence-Based Reading and Writing Section + Math Section clean",
     "train-test clean",
    "School 1 Class Rank (Numeric) clean",
    "School 1 Class Size (Numeric) clean"], axis = 1)
#For y_train2 I only kept Decision clean because it is what we are predicting 
y_train2 = Train3_clean['Decision clean']


## Simple Decision Tree (after data refinement method 1 with validation)

In [ ]:
X_train2, X_val2, y_train2, y_val2 = train_test_split(X_train2, y_train2, test_size=0.2, random_state=0)

In [ ]:
dtc2 = DecisionTreeClassifier(criterion='gini', random_state = 0)
dtc2.fit(X_train2,y_train2)

# The y's of the validation and test sets are being predicted using the training model 
y_train2_pred = dtc2.predict(X_train2)
y_val2_pred = dtc2.predict(X_val2)


#The accuracy, precision, recall, f1, and cohen kappa scores for the train set are calculated 
print("sp_train_Accuracy:", accuracy_score(y_train2, y_train2_pred))
print("sp_train_Precision:", precision_score(y_train2, y_train2_pred))
print("sp_train_Recall:", recall_score(y_train2, y_train2_pred))
print("sp_train_F1-score:", f1_score(y_train2, y_train2_pred))
print("Cohen's Kappa Score_train:", cohen_kappa_score(y_train2, y_train2_pred))

#The accuracy, precision, recall, f1, and cohen kappa scores for the validation set are calculated 
print("\nsp_val_Accuracy:", accuracy_score(y_val2, y_val2_pred))
print("sp_val_Precision:", precision_score(y_val2, y_val2_pred))
print("sp_val_Recall:", recall_score(y_val2, y_val2_pred))
print("sp_val_F1-score:", f1_score(y_val2, y_val2_pred))
print("Cohen's Kappa Score_test:", cohen_kappa_score(y_val2, y_val2_pred))


In [ ]:
values, counts = np.unique(y_val2_pred, return_counts=True)

print(dict(zip(values, counts)))

In [ ]:
feature_importances = dtc2.feature_importances_
feature_importances

In [ ]:
feature_importance_dict = {'feature_name': X_train2.columns, 'feature_importances': feature_importances}
feature_importance_df = pd.DataFrame(feature_importance_dict).sort_values(by='feature_importances', ascending=False)
print(feature_importance_df)

## Pruned Decision Tree (after data refinement method 1 with validation)

In [ ]:
#Shows how big the train, validation, and test datasets are 
print("Dataset Sizes:\n")

print(f"Training set (used to fit models): {X_train2.shape[0]} observations")
print(f"Validation set (used to select ccp_alpha): {X_val2.shape[0]} observations")

#Shows how many features are in the train, validation, and test sets 
print("\nFeature dimensions (number of columns):")
print(f"Training set: {X_train2.shape[1]} features")
print(f"Validation set: {X_val2.shape[1]} features")

In [ ]:
#This code is making a smiple decision tree
simple_tree2 = DecisionTreeClassifier(criterion='gini', #It uses gini to decide how to split 
                                     max_features=X_train2.shape[1], #It can use all the features within the dataset
                                     random_state=0, #The randomness has been set to 0
                                     min_samples_split=2, # Two observations are needed for a node to split 
                                     min_samples_leaf=1, #One observation is needed for a leaf node 
                                     max_depth=None, #There is no max depth
                                     ccp_alpha=0) # Should not be pruned since it is set to 0

# Looks for the best ccp_alpha values when pruning the tree 
#When the tree is pruned, a new ccp_alpha value is assigned to it
path = simple_tree2.cost_complexity_pruning_path(X_train2, y_train2)                                               
alphas = path['ccp_alphas']
impurities = path['impurities']

In [ ]:
#Stores results of pruned trees made in the loop
pruning_results2 = []


#Looks through all the ccp_alphas and uses that to train many different trees 
for alpha, impurity in zip(alphas, impurities):
    pruned_tree2 = DecisionTreeClassifier(criterion='gini', #It uses gini to decide how to split 
                                         max_features=X_train2.shape[1],#It can use all the features within the dataset
                                         random_state=0,#The randomness has been set to 0
                                         min_samples_split=2,# Two observations are needed for a node to split 
                                         min_samples_leaf=1,#One observation is needed for a leaf node
                                         max_depth=None, #There is no max depth
                                         ccp_alpha=alpha)# set to the alpha we found in the last cell
#pruned_tree2 is being fit to the train3 data 
    pruned_tree2.fit(X_train2, y_train2)
#Takes note of the ccp_alpa, impurity, number of leaves, and depth of each pruned tree
    pruning_results2.append({
        'ccp_alpha': alpha,
        'total_impurity': impurity,
        'n_leaves': pruned_tree2.get_n_leaves(),
        'max_depth': pruned_tree2.get_depth()
    })

# Makes the results into a dataframe 
pruning_results2_df = pd.DataFrame(pruning_results2)
#prints part of the dataframe 
pd.set_option('display.max_rows', 20)
pruning_results2_df.head(10)

In [ ]:
# Three lists were made for each dataset to save the Cohen's kappa scores from the loop
ACC_Train2_simple = []
ACC_Validation2_simple = []
# loop over the alpha scores and find the Cohen's kappa score for each of the sets 
for i in alphas:
    simple_tree2 = DecisionTreeClassifier(criterion='gini',#It uses gini to decide how to split 
                                         max_features=X_train2.shape[1],#It can use all the features within the dataset
                                         random_state=0,#The randomness has been set to 0
                                         min_samples_split=2,# Two observations are needed for a node to split 
                                         min_samples_leaf=1,#One observation is needed for a leaf node 
                                         max_depth=None,#There is no max depth
                                         ccp_alpha=i)# This is what the loop is iterating over
    #Fits the tree using train3 and simple_tree2 
    simple_tree2.fit(X_train2, y_train2)
    #The y's are being predicted using simple_tree2 
    y_train2_sub_pred = simple_tree2.predict(X_train2)
    y_val2_sub_pred = simple_tree2.predict(X_val2)

    #The cohen kappa scores are being appended to the lists 
    ACC_Train2_simple.append(cohen_kappa_score(y_train2, y_train2_sub_pred))
    ACC_Validation2_simple.append(cohen_kappa_score(y_val2, y_val2_sub_pred))


In [ ]:
#finds the best validation score for the validation set and sees what alpha score corresponds to that
max_acc_alpha_val2 = alphas[ACC_Validation2_simple.index(max(ACC_Validation2_simple))]
max_acc_alpha_val2

In [ ]:
#Makes a pruned decision tree based on the best Cohen's kappa alpha from the test set
pruned_tree2 = DecisionTreeClassifier(criterion='gini', #It uses gini to decide how to split 
                                     min_samples_split=2, # Two observations are needed for a node to split 
                                     min_samples_leaf=1,#One observation is needed for a leaf node 
                                     max_depth=None, #There is no max depth
                                     max_features=X_train2.shape[1],#It can use all the features within the dataset
                                     random_state=0,#The randomness has been set to 0
                                     ccp_alpha=max_acc_alpha_val2) #the alpha is set to the best max acc aplha that we found
#The pruned tree is being fit to the train3 data
pruned_tree2.fit(X_train2, y_train2)

# y values for the train, validation, and test sets are predicted using the trained pruned tree
y_train2_pred_pt = pruned_tree2.predict(X_train2)
y_val2_pred_pt = pruned_tree2.predict(X_val2)


In [ ]:
#The accuracy, precision, recall, f1, and cohen kappa scores for the train set are calculated 
print("pt_train_Accuracy:", accuracy_score(y_train2, y_train2_pred_pt))
print("pt_train_Precision:", precision_score(y_train2, y_train2_pred_pt))
print("pt_train_Recall:", recall_score(y_train2, y_train2_pred_pt))
print("pt_train_F1-score:", f1_score(y_train2, y_train2_pred_pt))
print("Cohen's Kappa Score_train:", cohen_kappa_score(y_train2, y_train2_pred_pt))

#The accuracy, precision, recall, f1, and cohen kappa scores for the validation set are calculated 
print("\npt_val_Accuracy:", accuracy_score(y_val2, y_val2_pred_pt))
print("pt_val_Precision:", precision_score(y_val2, y_val2_pred_pt))
print("pt_val_Recall:", recall_score(y_val2, y_val2_pred_pt))
print("pt_val_F1-score:", f1_score(y_val2, y_val2_pred_pt))
print("Cohen's Kappa Score_test:", cohen_kappa_score(y_val2, y_val2_pred_pt))


In [ ]:
#The feature importance scores for each column are calculated 
pt_feature_importances = pruned_tree2.feature_importances_  

#A dictionary with the feature importance scores is made with the corresponding column names 
feature_importance_dict = {
    'feature_name': X_train2.columns,  
    'pt_feature_importances': pt_feature_importances  
}
#The dictionary is made into a dataframe and sorted by highest fetaure importance to lowest 
feature_importance_df = pd.DataFrame(feature_importance_dict).sort_values(by='pt_feature_importances', ascending=False)  # Sort by importance

print(feature_importance_df)  

## Random Forest (after data refinement method 1 with validation)

In [ ]:
#The sizes of each of the datasets are printed 
print("Training set shape:", X_train2.shape)
print("Validation set shape:", X_val2.shape)


In [ ]:
#A Random forest is being made with these criteria 
Tree_RF2 = RandomForestClassifier(
    n_estimators=200, #There should be at least 200 trees in the forest 
    criterion='gini', #Gini is used to split 
    max_features=6, #There can not be more than 6 features per split 
    bootstrap=True, # gets the trees training sample with replacement
    max_samples=X_train2.shape[0], #can get a sample size that is the same size as the training set 
    max_depth=None, #There is no max depth 
    oob_score=True, #calculates OOB accuracy 
    random_state=0 #randomness is set to 0 
)
#The random forest is fit using the train3 data 
Tree_RF2.fit(X_train2, y_train2)

# Calculates the OOB accuracy 
print("Initial OOB Accuracy:", round(Tree_RF2.oob_score_, 3))

In [ ]:
#empty lists for the validation kappa and max features 
max2_features = []
val2_kappa_rf = []
#This loop finds the best amount of max features for the random forest using these criteria
for i in range(1, X_train2.shape[1] + 1):
    Tree_RF2 = RandomForestClassifier(
        n_estimators=100,  #There should be at least 100 trees in the forest                           
        criterion='gini',   #Gini is used to split             
        max_features=i,   #this is what the loop is iterating over          
        bootstrap=True,    # gets the trees training sample with replacement             
        max_samples=X_train2.shape[0],  #can get a sample size that is the same size as the training set 
        max_depth=None,  #There is no max depth                     
        oob_score=True,  #calculates OOB accuracy                 
        random_state=0    #randomness is set to 0                    
    )
    #The random forest is being fit to the train2 data
    Tree_RF2.fit(X_train2, y_train2)
    #The yal_val is being predicted using the random forest
    y_val2_pred = Tree_RF2.predict(X_val2)
       #The max features have been added to the list 
    max2_features.append(i)
    #The validation Cohen's kappa scores have been added to the list 
    val2_kappa_rf.append(cohen_kappa_score(y_val2, y_val2_pred))
   

# A data frame of the max features and their corresponding Cohen's kappa is made and printed 
#It is sorted from highest to lowest 
df_kappa_val2_rf = pd.DataFrame({
    'max_features': max2_features,
    'validation_kappa_rf': val2_kappa_rf
}).sort_values(by='validation_kappa_rf', ascending=False)

print(df_kappa_val2_rf.round(3))
# The best feature (the first one) is stored in best_max_val_features 
best_max_val2_features = int(df_kappa_val2_rf.iloc[0]['max_features'])
print("Optimal max_features based on validation kappa:", best_max_val2_features)

In [ ]:
#The best max feature is used to make another random forest 
Tree_RF2_Best = RandomForestClassifier(
    n_estimators=100,                    #There should be at least 100 trees in the forest 
    criterion='gini',                    #Gini is used to split 
    max_features=best_max_val2_features,      #max fetaures is set to the best max features that we found 
    bootstrap=True,                      #  gets the trees training sample with replacement 
    max_samples=X_train2.shape[0],       #can get a sample size that is the same size as the training set 
    max_depth=None,                      #There is no max depth               
    oob_score=True,                      #calculates OOB accuracy   
    random_state=0                       #randomness is set to 0      
)       
#The random forest is fit using the train3 data
Tree_RF2_Best.fit(X_train2, y_train2)

# # the y's are predicted using the fitted model 
y_train2_pred_RF = Tree_RF2_Best.predict(X_train2)
y_val2_pred_RF = Tree_RF2_Best.predict(X_val2)


#  Shows the oob prediction summary from the random forest 
oob_accuracy_RF2 = Tree_RF2_Best.oob_score_


In [ ]:
#This function gives you the accuracy, precision, recall, f1, and cohen kappa score for both sets
def report_metrics(y_true, y_pred, label):
    print(label + "_Accuracy:", round(accuracy_score(y_true, y_pred), 3))
    print(label + "_Precision:", round(precision_score(y_true, y_pred), 3))
    print(label + "_Recall:", round(recall_score(y_true, y_pred), 3))
    print(label + "_F1_score:", round(f1_score(y_true, y_pred), 3))
    print(label + "_Kappa:", round(cohen_kappa_score(y_true, y_pred), 3))
    print()
#The performance metrics for train and validation are printed 
report_metrics(y_train2, y_train2_pred_RF, "RF_train")
report_metrics(y_val2, y_val2_pred_RF, "RF_validation")
#The OOB accuracy is printed and compared to the validation  accuracy 
print("RF_OOB_Accuracy:", round(oob_accuracy_RF2, 3))
print("Validation vs OOB Accuracy Difference:", round(accuracy_score(y_val2, y_val2_pred_RF) - oob_accuracy_RF2, 3))




In [ ]:
#The feature importance scores for each column is calculated with the best random forest model  
feature_importances = Tree_RF2_Best.feature_importances_

#A dicitonary with the column names and each columns feature importance score is made 
feature_importance_dict = {
    'feature_name': X_train2.columns,
    'feature_importances': feature_importances
}
#That dictionary is then converted into a dataframe and is printed in order from highest feature importance to lowest 
feature_importance_df = pd.DataFrame(feature_importance_dict).sort_values(
    by='feature_importances', ascending=False
)
pd.set_option('display.max_rows', None)
print(feature_importance_df.round(3))

## Data Refinement Method 2 (Make a new column for Merit Award that shows if a student received money from Trinity or not)

In [ ]:
#I am making a copy of Train
Train4 = Train.copy()

I believe that this will improve the performance metrics of my model because it further simplifies the merit award column. If a student recieves money from the university that they applied to, they are obviously more likely to accept their admission. 

In [ ]:
#I am recategorizing Merit Award by Yes for students who received money from Trinity and no for those who didn't 
Train4["Merit Award clean"].value_counts()
has_recieved_money = ["University Merit", "International", "Tution Waived"]
Train4["Merit Award clean"] = Train4["Merit Award clean"] .apply(lambda x: "Yes" if x in has_recieved_money else "No")
#I am checking to makre sure it worked by looking at the value counts for each catgeory within Merit Award clean 
Train4["Merit Award clean"].value_counts()

In [ ]:
#This function label all of the catgeorical columns in Train4
def encode(Train4): 
    # It loops through all the columns and sees which ones are objects and which ones are categories  
    for col in Train4.columns:                       
        if Train4[col].dtype == 'object' or Train4[col].dtype == 'category':  
            #Then it encodes each column that meets the criteria 
            le = LabelEncoder()                  
            Train4[col] = le.fit_transform(Train4[col]) 
    return Train4                                   
#The function was called and saved back to the original Train dataframe
Train4 = encode(Train4)  

In [ ]:
#I am double-checking to make sure Merit Award was label encoded 
Train4["Merit Award clean"].value_counts()

In [ ]:
# I made a clean copy of Train4 that I will modify
Train4_clean = Train4.copy()
#fThis for loop looks through all of the columns in Train4_clean and drops the ones that do not have clean in their name
for col in Train4_clean.columns:
    if 'clean' not in col:
        Train4_clean=Train4_clean.drop(columns=col)
#For X_train3 I dropped the columns that we do not need for the decision trees and the column that we are predicting 
X_train3 = Train4_clean.drop(
    ["Decision clean",
     "ID clean",
     "SAT R Evidence-Based Reading and Writing Section + Math Section clean",
     "train-test clean",
    "School 1 Class Rank (Numeric) clean",
    "School 1 Class Size (Numeric) clean"], axis = 1)
#For y_train3 I only kept Decision clean because it is what we are predicting 
y_train3 = Train4_clean['Decision clean']

## Simple Decision Tree (after data refinement method 2 with validation)

In [ ]:
X_train3, X_val3, y_train3, y_val3 = train_test_split(X_train3, y_train3, test_size=0.2, random_state=0)

In [ ]:
# The train data is being used to fit the simple deicison tree 
dtc3 = DecisionTreeClassifier(criterion='gini', random_state = 0)
dtc3.fit(X_train3,y_train3)

# The y's of the validation and test sets are being predicted using the training model 
y_train3_pred = dtc3.predict(X_train3)
y_val3_pred = dtc3.predict(X_val3)

#The accuracy, precision, recall, f1, and cohen kappa scores for the train set are calculated 
print("sp_train_Accuracy:", accuracy_score(y_train3, y_train3_pred))
print("sp_train_Precision:", precision_score(y_train3, y_train3_pred))
print("sp_train_Recall:", recall_score(y_train3, y_train3_pred))
print("sp_train_F1-score:", f1_score(y_train3, y_train3_pred))
print("Cohen's Kappa Score_train:", cohen_kappa_score(y_train3, y_train3_pred))

#The accuracy, precision, recall, f1, and cohen kappa scores for the validation set are calculated 
print("\nsp_val_Accuracy:", accuracy_score(y_val3, y_val3_pred))
print("sp_val_Precision:", precision_score(y_val3, y_val3_pred))
print("sp_val_Recall:", recall_score(y_val3, y_val3_pred))
print("sp_val_F1-score:", f1_score(y_val3, y_val3_pred))
print("Cohen's Kappa Score_test:", cohen_kappa_score(y_val3, y_val3_pred))

In [ ]:
values, counts = np.unique(y_val3_pred, return_counts=True)

print(dict(zip(values, counts)))

In [ ]:
feature_importances = dtc3.feature_importances_
feature_importances

In [ ]:
feature_importance_dict = {'feature_name': X_train3.columns, 'feature_importances': feature_importances}
feature_importance_df = pd.DataFrame(feature_importance_dict).sort_values(by='feature_importances', ascending=False)
print(feature_importance_df)

## Pruned Decision Tree (after data refinement method 2 with validation)

In [ ]:
#Shows how big the train, validation, and test datasets are 
print("Dataset Sizes:\n")

print(f"Training set (used to fit models): {X_train3.shape[0]} observations")
print(f"Validation set (used to select ccp_alpha): {X_val3.shape[0]} observations")


#Shows how many features are in the train, validation, and test sets 
print("\nFeature dimensions (number of columns):")
print(f"Training set: {X_train3.shape[1]} features")
print(f"Validation set: {X_val3.shape[1]} features")


In [ ]:
#This code is making a smiple decision tree
simple_tree3 = DecisionTreeClassifier(criterion='gini', #It uses gini to decide how to split 
                                     max_features=X_train3.shape[1], #It can use all the features within the dataset
                                     random_state=0, #The randomness has been set to 0
                                     min_samples_split=2, # Two observations are needed for a node to split 
                                     min_samples_leaf=1, #One observation is needed for a leaf node 
                                     max_depth=None, #There is no max depth
                                     ccp_alpha=0) # Should not be pruned since it is set to 0

# Looks for the best ccp_alpha values when pruning the tree
#When the tree is pruned, a new ccp_alpha value is assigned to it
path = simple_tree3.cost_complexity_pruning_path(X_train3, y_train3)                                               
alphas = path['ccp_alphas']
impurities = path['impurities']

In [ ]:
#Stores results of pruned trees made in the loop
pruning_results3 = []


#Looks through all the ccp_alphas and uses that to train many different trees 
for alpha, impurity in zip(alphas, impurities):
    pruned_tree3 = DecisionTreeClassifier(criterion='gini',#It uses gini to decide how to split 
                                         max_features=X_train3.shape[1],#It can use all the features within the dataset
                                         random_state=0,#The randomness has been set to 0
                                         min_samples_split=2,# Two observations are needed for a node to split 
                                         min_samples_leaf=1, #One observation is needed for a leaf node
                                         max_depth=None,#There is no max depth
                                         ccp_alpha=alpha) #set to alpah we found previously 
  #The pruned_tree3 is being fit to the train4 data  
    pruned_tree3.fit(X_train3, y_train3)
#Takes note of the ccp_alpa, impurity, number of leaves, and depth of each pruned tree
    pruning_results3.append({
        'ccp_alpha': alpha,
        'total_impurity': impurity,
        'n_leaves': pruned_tree3.get_n_leaves(),
        'max_depth': pruned_tree3.get_depth()
    })

# Makes the results into a dataframe 
pruning_results3_df = pd.DataFrame(pruning_results3)
#prints part of the dataframe 
pd.set_option('display.max_rows', 20)
pruning_results3_df.head(10)

In [ ]:
# Three lists were made for each dataset to save the Cohen's kappa scores from the loop
ACC_Train3_simple = []
ACC_Validation3_simple = []
# loop over the alpha scores and find the Cohen's kappa score for each of the sets 
for i in alphas:
    simple_tree3 = DecisionTreeClassifier(criterion='gini',#It uses gini to decide how to split 
                                         max_features=X_train3.shape[1],#It can use all the features within the dataset
                                         random_state=0,#The randomness has been set to 0
                                         min_samples_split=2,# Two observations are needed for a node to split 
                                         min_samples_leaf=1, #One observation is needed for a leaf node
                                         max_depth=None,#There is no max depth
                                         ccp_alpha=i) #this is what the loop is ierating over 
    #The simple_tree3 is being fit to the train4 data
    simple_tree3.fit(X_train3, y_train3)
    #The y's are being predicted using the simple_tree3
    y_train3_sub_pred = simple_tree3.predict(X_train3)
    y_val3_sub_pred = simple_tree3.predict(X_val3)
    #The cohen kappas are being added to the lists 
    ACC_Train3_simple.append(cohen_kappa_score(y_train3, y_train3_sub_pred))
    ACC_Validation3_simple.append(cohen_kappa_score(y_val3, y_val3_sub_pred))
   

In [ ]:
#finds the best validation score for the validation set and sees what alpha score corresponds to that
max_acc_alpha_val3 = alphas[ACC_Validation3_simple.index(max(ACC_Validation3_simple))]
max_acc_alpha_val3

In [ ]:
#Pruned_tree3 is being made according to these criteria which I have explained above 
#It is using the best max_acc_alpha that was found before 
pruned_tree3 = DecisionTreeClassifier(criterion='gini', #It uses gini to decide how to split 
                                     min_samples_split=2, # Two observations are needed for a node to split 
                                     min_samples_leaf=1, #One observation is needed for a leaf node
                                     max_depth=None,#There is no max depth
                                     max_features=X_train3.shape[1],#It can use all the features within the dataset
                                     random_state=0,#The randomness has been set to 0
                                     ccp_alpha=max_acc_alpha_val3) #set to the best best max acc alpha found earlier 
#pruned_tree3 is being fit to the train4 data 
pruned_tree3.fit(X_train3, y_train3)

# y values for the train, validation, and test sets are predicted using the trained pruned tree
y_train3_pred_pt = pruned_tree3.predict(X_train3)
y_val3_pred_pt = pruned_tree3.predict(X_val3)

In [ ]:
#The accuracy, precision, recall, f1, and cohen kappa scores for the train set are calculated 
print("pt_train_Accuracy:", accuracy_score(y_train3, y_train3_pred_pt))
print("pt_train_Precision:", precision_score(y_train3, y_train3_pred_pt))
print("pt_train_Recall:", recall_score(y_train3, y_train3_pred_pt))
print("pt_train_F1-score:", f1_score(y_train3, y_train3_pred_pt))
print("Cohen's Kappa Score_train:", cohen_kappa_score(y_train3, y_train3_pred_pt))

#The accuracy, precision, recall, f1, and cohen kappa scores for the validation set are calculated 
print("\npt_val_Accuracy:", accuracy_score(y_val3, y_val3_pred_pt))
print("pt_val_Precision:", precision_score(y_val3, y_val3_pred_pt))
print("pt_val_Recall:", recall_score(y_val3, y_val3_pred_pt))
print("pt_val_F1-score:", f1_score(y_val3, y_val3_pred_pt))
print("Cohen's Kappa Score_test:", cohen_kappa_score(y_val3, y_val3_pred_pt))


In [ ]:
#The feature importance scores for each column are calculated 
pt_feature_importances = pruned_tree3.feature_importances_  

#A dictionary with the feature importance scores is made with the corresponding column names 
feature_importance_dict = {
    'feature_name': X_train3.columns,  
    'pt_feature_importances': pt_feature_importances  
}
#The dictionary is made into a dataframe and sorted by highest fetaure importance to lowest 
feature_importance_df = pd.DataFrame(feature_importance_dict).sort_values(by='pt_feature_importances', ascending=False)  # Sort by importance

print(feature_importance_df)  

## Random Forest (after data refinement method 2 with validation)

In [ ]:
#The sizes of each of the datasets are printed 
print("Training set shape:", X_train3.shape)
print("Validation set shape:", X_val3.shape)


In [ ]:
#A Random forest is ebing made with these criteria 
Tree_RF3 = RandomForestClassifier(
    n_estimators=200, #There should be at least 200 trees in the forest 
    criterion='gini', #Gini is used to split 
    max_features=6, #There can not be more than 6 features per split 
    bootstrap=True, # gets the trees training sample with replacement
    max_samples=X_train3.shape[0], #can get a sample size that is the same size as the training set 
    max_depth=None, #There is no max depth 
    oob_score=True, #calculates OOB accuracy 
    random_state=0 #randomness is set to 0 
)
#The random forest is fit using the train data 
Tree_RF3.fit(X_train3, y_train3)

# Calculates the OOB accuracy 
print("Initial OOB Accuracy:", round(Tree_RF3.oob_score_, 3))

In [ ]:
#empty lists for the validation kappa and max features 
max_features3 = []
val3_kappa_rf = []
#This loop finds the best amount of max features for the random forest using these criteria
for i in range(1, X_train3.shape[1] + 1):
    Tree_RF3 = RandomForestClassifier(
        n_estimators=100,    #There should be at least 100 trees in the forest           
        criterion='gini',    #Gini is used to split            
        max_features=i,         #max fetaures is being determined by the loop         
        bootstrap=True,          # gets the trees training sample with replacement        
        max_samples=X_train3.shape[0],  #can get a sample size that is the same size as the training set 
        max_depth=None,     #There is no max depth                   
        oob_score=True,      #calculates OOB accuracy                  
        random_state=0         #randomness is set to 0                 
    )
    #The random forest is being fit to the train2 data 
    Tree_RF3.fit(X_train3, y_train3)
    #The yal_val is being predicted using the random forest
    y_val3_pred = Tree_RF3.predict(X_val3)
   #The max features have been added to the list 
    max_features3.append(i)
#The validation Cohen's kappa scores have been added to the list     
    val3_kappa_rf.append(cohen_kappa_score(y_val3, y_val3_pred))
   

# A data frame of the max features and their corresponding Cohen's kappa is made and printed 
#It is sorted from highest to lowest
df_kappa_val3_rf = pd.DataFrame({
    'max_features': max_features3,
    'validation_kappa_rf': val3_kappa_rf
}).sort_values(by='validation_kappa_rf', ascending=False)

print(df_kappa_val3_rf.round(3))
# The best feature (the first one) is stored in best_max_val_features 
best_max_val3_features = int(df_kappa_val3_rf.iloc[0]['max_features'])
print("Optimal max_features based on validation kappa:", best_max_val3_features)


In [ ]:
#The best max feature is used to make another random forest 
Tree_RF3_Best = RandomForestClassifier(
    n_estimators=100,  #There should be at least 100 trees in the forest                       
    criterion='gini',   #Gini is used to split                   
    max_features=best_max_val3_features,  #uses the best max features found above         
    bootstrap=True,   # gets the trees training sample with replacement                    
    max_samples=X_train3.shape[0],  #can get a sample size that is the same size as the training set       
    max_depth=None,    #There is no max depth                            
    oob_score=True,    #calculates OOB accuracy                    
    random_state=0        #randomness is set to 0                          
)
#The random forest is fitted on the train4 data 
Tree_RF3_Best.fit(X_train3, y_train3)

# The y's are predicted using the fitted model
y_train3_pred_RF = Tree_RF3_Best.predict(X_train3)
y_val3_pred_RF = Tree_RF3_Best.predict(X_val3)

# Shows the oob prediction summary from the random forest 
oob_accuracy_RF3 = Tree_RF3_Best.oob_score_


In [ ]:
#This function gives you the accuracy, precision, recall, f1, and cohen kappa
def report_metrics(y_true, y_pred, label):
    print(label + "_Accuracy:", round(accuracy_score(y_true, y_pred), 3))
    print(label + "_Precision:", round(precision_score(y_true, y_pred), 3))
    print(label + "_Recall:", round(recall_score(y_true, y_pred), 3))
    print(label + "_F1_score:", round(f1_score(y_true, y_pred), 3))
    print(label + "_Kappa:", round(cohen_kappa_score(y_true, y_pred), 3))
    print()
#The performance metrics for train and validation are printed 
report_metrics(y_train3, y_train3_pred_RF, "RF_train")
report_metrics(y_val3, y_val3_pred_RF, "RF_validation")
#The OOB accuracy is printed and compared to the validation  accuracy 
print("RF_OOB_Accuracy:", round(oob_accuracy_RF3, 3))
print("Validation vs OOB Accuracy Difference:", round(accuracy_score(y_val3, y_val3_pred_RF) - oob_accuracy_RF3, 3))




In [ ]:
#The feature importance scores for each column is calculated with the best random forest model  
feature_importances = Tree_RF3_Best.feature_importances_

#A dicitonary with the column names and each columns feature importance score is made 
feature_importance_dict = {
    'feature_name': X_train3.columns,
    'feature_importances': feature_importances
}
#That dictionary is then converted into a dataframe and is printed in order from highest feature importance to lowest 

feature_importance_df = pd.DataFrame(feature_importance_dict).sort_values(
    by='feature_importances', ascending=False
)
pd.set_option('display.max_rows', None)
print(feature_importance_df.round(3))

## Test Set 

## Best Simple Decision Tree

 The best performing simple decision tree was the one with the first data refinement because it has a cohen kappa score of 0.335.

In [ ]:
# I made a copy of Test
Test3 = Test.copy()
#I recategorized School 1 GPA Recalculated clean into excellent, goodm and bad gpas 
Test3["School 1 GPA Recalculated clean"] = Test3["School 1 GPA Recalculated clean"].apply(lambda x: 'Good' if x in good else x)
Test3["School 1 GPA Recalculated clean"] = Test3["School 1 GPA Recalculated clean"].apply(lambda x: 'Excellent' if x in excellent else x)
Test3["School 1 GPA Recalculated clean"] = Test3["School 1 GPA Recalculated clean"].apply(lambda x: 'Bad' if x in bad else x)
#I printed value counts to make sure it worked 
print(Test3["School 1 GPA Recalculated clean"].value_counts())
#I label encoded it 
le = LabelEncoder()
Test3["School 1 GPA Recalculated clean"] = le.fit_transform(Test3["School 1 GPA Recalculated clean"])
#I printed value counts again to make sure the label encoding worked
print(Test3["School 1 GPA Recalculated clean"].value_counts())

In [ ]:
# This function one-hot encodes the categorical columns within Test
def encode(Test3):  
    # It loops through all the columns and sees which ones are objects and which ones are categories  
    for col in Test3.columns:     
        if Test3[col].dtype == 'object' or Test3[col].dtype == 'category':  
            #Then it encodes each column that meets the criteria 
            le = LabelEncoder()                  
            Test3[col] = le.fit_transform(Test3[col]) 
    return Test3                                   
#The function was called and saved back to the original Test dataframe
Test3 = encode(Test3)  

In [ ]:
# I made a copy of Test3 and called Test3_clean
Test3_clean = Test3.copy()
# For all the columns in Test3_clean I looked for the ones that do not have clean in their name and dropped them 
for col in Test3_clean.columns:
    if 'clean' not in col:
        Test3_clean=Test3_clean.drop(columns=col)
#I checked to see if only clean columns remained
Test3_clean.info()
#For X I dropped decision clean, ID clean, SAT R Evidence-Based Reading and Writing Section + Math Section clean, train-test clean,School 1 Class Rank (Numeric) clean, School 1 Class Size (Numeric) clean 
X_test = Test3_clean.drop(
    ["Decision clean",
     "ID clean",
     "SAT R Evidence-Based Reading and Writing Section + Math Section clean",
     "train-test clean",
    "School 1 Class Rank (Numeric) clean",
    "School 1 Class Size (Numeric) clean"], axis = 1)
#For y I only kept Decision clean because it is what we are predicting 
y_test = Test3_clean['Decision clean']

In [ ]:
# The train data is being used to fit the simple decision tree and the splits will be decided with gini impurity 
#There is a random state of 0
dtc = DecisionTreeClassifier(criterion='gini', random_state = 0)
#I am using X_train2 and y_train2 because that is what I used for simple tree(data refinement method 1)
dtc.fit(X_train2,y_train2)


# The y's of the train and test sets are being predicted using the training model 
y_train2_pred = dtc.predict(X_train2)
y_test_pred = dtc.predict(X_test)

#The accuracy, precision, recall, f1, and cohen kappa scores for the train set are calculated 
print("sp_train_Accuracy:", accuracy_score(y_train2, y_train2_pred))
print("sp_train_Precision:", precision_score(y_train2, y_train2_pred))
print("sp_train_Recall:", recall_score(y_train2, y_train2_pred))
print("sp_train_F1-score:", f1_score(y_train2, y_train2_pred))
print("Cohen's Kappa Score_train:", cohen_kappa_score(y_train2, y_train2_pred))

#The accuracy, precision, recall, f1, and cohen kappa scores for the test set are calculated 
print("\nsp_val_Accuracy:", accuracy_score(y_test, y_test_pred))
print("sp_val_Precision:", precision_score(y_test, y_test_pred))
print("sp_val_Recall:", recall_score(y_test, y_test_pred))
print("sp_val_F1-score:", f1_score(y_test, y_test_pred))
print("Cohen's Kappa Score_test:", cohen_kappa_score(y_test, y_test_pred))

In [ ]:
#Shows the number of predicted class labels in the test set 
values, counts = np.unique(y_test_pred, return_counts=True)

print(dict(zip(values, counts)))

In [ ]:
#Shows the feature importance of each column
#Numbers are in the order of the columns in the dataset 
feature_importances = dtc.feature_importances_
feature_importances

In [ ]:
# Shows the feature importance of each column in order of how important they are 
feature_importance_dict = {'feature_name': X_train2.columns, 'feature_importances': feature_importances}
feature_importance_df = pd.DataFrame(feature_importance_dict).sort_values(by='feature_importances', ascending=False)
print(feature_importance_df)

## Best Pruned Decision Tree

The pruned decision tree with the best performance is the one with the second data modification because it has a cohen kappa score of 0.44.

In [ ]:
#I am making a copy of Test
Test4 = Test.copy()
Test4["Merit Award clean"].value_counts()
#I am recategorizing Merit Award by Yes for students who received money from Trinity and no for those who didn't 
has_recieved_money = ["University Merit", "International", "Tution Waived"]
Test4["Merit Award clean"] = Test4["Merit Award clean"] .apply(lambda x: "Yes" if x in has_recieved_money else "No")
# I am printing value counts to make sure it worked 
Test4["Merit Award clean"].value_counts()


In [ ]:
# This function one-hot encodes the categorical columns within Test
def encode(Test4):  
    # It loops through all the columns and sees which ones are objects and which ones are categories  
    for col in Test4.columns:     
        if Test4[col].dtype == 'object' or Test4[col].dtype == 'category':  
            #Then it encodes each column that meets the criteria 
            le = LabelEncoder()                  
            Test4[col] = le.fit_transform(Test4[col]) 
    return Test4                                  
#The function was called and saved back to the original Test dataframe
Test4 = encode(Test4)  

In [ ]:
# I made a cleaned copy of Test
Test4_clean = Test4.copy()
# For all the columns in Test4_clean I looked for the ones that do not have clean in their name and dropped them 
for col in Test4_clean.columns:
    if 'clean' not in col:
        Test4_clean=Test4_clean.drop(columns=col)
#I checked to see if only clean columns remained
Test4_clean.info()
#For X I dropped decision clean, ID clean, SAT R Evidence-Based Reading and Writing Section + Math Section clean, train-test clean,School 1 Class Rank (Numeric) clean, School 1 Class Size (Numeric) clean 
X_test2 = Test4_clean.drop(
    ["Decision clean",
     "ID clean",
     "SAT R Evidence-Based Reading and Writing Section + Math Section clean",
     "train-test clean",
    "School 1 Class Rank (Numeric) clean",
    "School 1 Class Size (Numeric) clean"], axis = 1)
#For y I only kept Decision clean because it is what we are predicting 
y_test2 = Test4_clean['Decision clean']

In [ ]:
# I am printing value counts to make sure it was encoded 
Test4["Merit Award clean"].value_counts()

In [ ]:
#Shows how big the train and validation and train data sets 
print("Dataset Sizes:\n")
#I am using X_train3 and X_val3 since those are the ones I used for the pruned tree (data refinement method 2)
print(f"Training set (used to fit models): {X_train3.shape[0]} observations")
print(f"Validation set (used to select ccp_alpha): {X_val3.shape[0]} observations")
print(f"Validation set (used to select ccp_alpha): {X_test2.shape[0]} observations")
#Shows how many features are in the train and validation and train data sets 
print("\nFeature dimensions (number of columns):")
print(f"Training set: {X_train3.shape[1]} features")
print(f"Validation set: {X_val3.shape[1]} features")
print(f"Validation set: {X_test2.shape[1]} features")

In [ ]:
#This code is making a smiple decision tree
simple_tree4 = DecisionTreeClassifier(criterion='gini', #It uses gini to decide how to split 
                                     max_features=X_train3.shape[1], #It can use all the features within the dataset
                                     random_state=0, #The randomness has been set to 0
                                     min_samples_split=2, # Two observations are needed for a node to split 
                                     min_samples_leaf=1, #One observation is needed for a leaf node 
                                     max_depth=None, #There is no max depth
                                     ccp_alpha=0) # Should not be pruned since it is set to 0

# Looks for the best ccp_alpha values when pruning the tree
#When the tree is pruned, a new ccp_alpha value is assigned to it
path = simple_tree4.cost_complexity_pruning_path(X_train3, y_train3)                                               
alphas = path['ccp_alphas']
impurities = path['impurities']

In [ ]:
#Stores results of pruned trees made in the loop
pruning_results4 = []

# Loop through each alpha value and train a tree using that pruning strength.
for alpha, impurity in zip(alphas, impurities):
    pruned_tree4 = DecisionTreeClassifier(criterion='gini',#It uses gini to decide how to split 
                                         max_features=X_train3.shape[1],#It can use all the features within the dataset
                                         random_state=0,#The randomness has been set to 0
                                         min_samples_split=2,# Two observations are needed for a node to split 
                                         min_samples_leaf=1,#One observation is needed for a leaf node 
                                         max_depth=None,#There is no max depth
                                         ccp_alpha=alpha) # set to the alpha we found previously  
    #The pruned_tree3 is being fit to the train4 data
    pruned_tree4.fit(X_train3, y_train3)
#Takes note of the ccp_alpa, impurity, number of leaves, and depth of each pruned tree
    pruning_results4.append({
        'ccp_alpha': alpha,
        'total_impurity': impurity,
        'n_leaves': pruned_tree4.get_n_leaves(),
        'max_depth': pruned_tree4.get_depth()
    })

# Makes the results into a dataframe 
pruning_results_df = pd.DataFrame(pruning_results4)
#prints part of the dataframe 
pd.set_option('display.max_rows', 200)
pruning_results_df.head(10)

In [ ]:
# Three lists were made for each dataset to save the Cohen's kappa scores from the loop
ACC_Train4_simple = []
ACC_Validation4_simple = []
# loop over the alpha scores and find the Cohen's kappa score for each of the sets 
for i in alphas:
    simple_tree4 = DecisionTreeClassifier(criterion='gini',#It uses gini to decide how to split 
                                         max_features=X_train3.shape[1],#It can use all the features within the dataset
                                         random_state=0,#The randomness has been set to 0
                                         min_samples_split=2,# Two observations are needed for a node to split 
                                         min_samples_leaf=1,#One observation is needed for a leaf node 
                                         max_depth=None,#There is no max depth
                                         ccp_alpha=i)# this is what the loop is iterating over 
    #The simple_tree4 is being fit to the train4 data
    simple_tree4.fit(X_train3, y_train3)
    #The y's are being predicted using the simple_tree4
    y_train3_sub_pred = simple_tree4.predict(X_train3)
    y_val3_sub_pred = simple_tree4.predict(X_val3)
    #The cohen kappas are being added to the lists 
    ACC_Train4_simple.append(cohen_kappa_score(y_train3, y_train3_sub_pred))
    ACC_Validation4_simple.append(cohen_kappa_score(y_val3, y_val3_sub_pred))

In [ ]:
# We can return the index for the maximum validation Cohen's kappa score and then find the corresponding alpha.
max_acc_alpha4 = alphas[ACC_Validation4_simple.index(max(ACC_Validation4_simple))]
max_acc_alpha4

In [ ]:
# Use the best alpha to construct the pruned tree.
# After choosing the best alpha with the validation set, we fit the final pruned tree on the full training data.
pruned_tree4 = DecisionTreeClassifier(criterion='gini', #It uses gini to decide how to split 
                                     min_samples_split=2, # Two observations are needed for a node to split 
                                     min_samples_leaf=1,#One observation is needed for a leaf node 
                                     max_depth=None,#There is no max depth
                                     max_features=X_train3.shape[1],
                                     random_state=0,#The randomness has been set to 0
                                     ccp_alpha=max_acc_alpha4)#sets the alpha to the best one that we found previously  
                                      
#pruned_tree3 is being fit to the train4 data                                     
pruned_tree4.fit(X_train3, y_train3)

# y values for the train, validation, and test sets are predicted using the trained pruned tree
y_train3_pred_pt = pruned_tree4.predict(X_train3)
y_test2_pred_pt = pruned_tree4.predict(X_test2)

In [ ]:
#The accuracy, precision, recall, f1, and cohen kappa scores for the train set are calculated 
print("pt_train_Accuracy:", accuracy_score(y_train3, y_train3_pred_pt))
print("pt_train_Precision:", precision_score(y_train3, y_train3_pred_pt))
print("pt_train_Recall:", recall_score(y_train3, y_train3_pred_pt))
print("pt_train_F1-score:", f1_score(y_train3, y_train3_pred_pt))
print("Cohen's Kappa Score_train:", cohen_kappa_score(y_train3, y_train3_pred_pt))

#The accuracy, precision, recall, f1, and cohen kappa scores for the validation set are calculated 
print("\npt_test_Accuracy:", accuracy_score(y_test2, y_test2_pred_pt))
print("pt_test_Precision:", precision_score(y_test2, y_test2_pred_pt))
print("pt_test_Recall:", recall_score(y_test2, y_test2_pred_pt))
print("pt_test_F1-score:", f1_score(y_test2, y_test2_pred_pt))
print("Cohen's Kappa Score_test:", cohen_kappa_score(y_test2, y_test2_pred_pt))

In [ ]:
#The feature importance scores for each column are calculated
pt_feature_importances = pruned_tree4.feature_importances_  # Extract feature importances from the trained pruned tree

#A dictionary with the feature importance scores is made with the corresponding column names 
feature_importance_dict = {
    'feature_name': X_train3.columns,  
    'pt_feature_importances': pt_feature_importances  
}
#The dictionary is made into a dataframe and sorted by highest feature importance to lowest 
feature_importance_df = pd.DataFrame(feature_importance_dict).sort_values(by='pt_feature_importances', ascending=False)  # Sort by importance

print(feature_importance_df)  

In [ ]:
#I am importing matplotlib.pyplot to make a bar chart 
import matplotlib.pyplot as plt
#I am saving the top 10 features into top_10_feature_importance
top_10_feature_importances = feature_importance_df.head(10).sort_values(by="pt_feature_importances", ascending=True)
#I am making a bar chart with the top 10 fetaures for my memo
plt.barh(top_10_feature_importances["feature_name"], top_10_feature_importances["pt_feature_importances"], color='pink')

## Best Random Forest

The random forest with the best performance was with the one with the second data refinement which has a cohen kappa score of 0.51.

In [ ]:
#The sizes of each of the datasets are printed 
# I am using X_train3 and X_val3 because that is what I used for the random forest with the second data refinement 
#I am using X_test2 because it is the same data refinement method 2 used in the best pruned tree above so we can uses the same x_train2 defined for that
print("Training set shape:", X_train3.shape)
print("Validation set shape:", X_val3.shape)
print("Test set shape:", X_test2.shape)


In [ ]:
#A Random forest is ebing made with these criteria 
Tree_RF4 = RandomForestClassifier(
    n_estimators=200,#There should be at least 200 trees in the forest
    criterion='gini', #Gini is used to split 
    max_features=6,#There can not be more than 6 features per split 
    bootstrap=True, # gets the trees training sample with replacement
    max_samples=X_train3.shape[0],#can get a sample size that is the same size as the training set 
    max_depth=None, #There is no max depth 
    oob_score=True,#calculates OOB accuracy 
    random_state=0 #randomness is set to 0 
)
#The random forest is fit using the train data 
Tree_RF4.fit(X_train3, y_train3)

# Calculates the OOB accuracy which is an internal performance score that is calculated using the oob_score
print("Initial OOB Accuracy:", round(Tree_RF4.oob_score_, 3))

In [ ]:
#empty lists for the validation kappa and max features 
max_features4 = []
val_kappa_rf4 = []
#This loop finds the best features for the random forest using these criteria
for i in range(1, X_train3.shape[1] + 1):
    Tree_RF4 = RandomForestClassifier(
        n_estimators=100,             #There should be at least 100 trees in the forest        
        criterion='gini',              #Gini is used to split
        max_features=i,                #max fetaures is being determined by the loop           
        bootstrap=True,               # gets the trees training sample with replacement
        max_samples=X_train3.shape[0],  #can get a sample size that is the same size as the training set 
        max_depth=None,                #There is no max depth      
        oob_score=True,                #calculates OOB accuracy       
        random_state=0                #randomness is set to 0      
    )
    #The random forest is being fit to the train2 data 
    Tree_RF4.fit(X_train3, y_train3)
    #The yal_val is being predicted using the random forest
    y_val3_pred = Tree_RF4.predict(X_val3)
    #The max features have been added to the list 
    max_features4.append(i)
    #The validation Cohen's kappa scores have been added to the list
    val_kappa_rf4.append(cohen_kappa_score(y_val3, y_val3_pred))

# A data frame of the max features and their corresponding Cohen's kappa is made and printed 
#It is sorted from highest to lowest 
df_kappa_rf4 = pd.DataFrame({
    'max_features': max_features4,
    'validation_kappa_rf': val_kappa_rf4
}).sort_values(by='validation_kappa_rf', ascending=False)

print(df_kappa_rf4.round(3))
# The best feature (the first one) is stored in best_max_val_features 
best_max_features4 = int(df_kappa_rf4.iloc[0]['max_features'])
print("Optimal max_features based on validation kappa:", best_max_features4)


In [ ]:
#A Random forest is being made with these criteria 
Tree_RF_Best4 = RandomForestClassifier(
    n_estimators=100,                    #There should be at least 100 trees in the forest 
    criterion='gini',                     #Gini is used to split 
    max_features=best_max_features4,      #max fetaures is set to the best max features we found 
    bootstrap=True,                      # gets the trees training sample with replacement
    max_samples=X_train3.shape[0],        #can get a sample size that is the same size as the training set 
    max_depth=None,                      #There is no max depth 
    oob_score=True,                      #calculates OOB accuracy
    random_state=0                       #randomness is set to 0 
)
#The random forest is fit using the train data 
Tree_RF_Best4.fit(X_train3, y_train3)

# the y's are predicted using the fitted model 
y_train3_pred_RF = Tree_RF_Best4.predict(X_train3)
y_val3_pred_RF = Tree_RF_Best4.predict(X_val3)
y_test2_pred_RF = Tree_RF_Best4.predict(X_test2)

# Shows the oob prediction summary from the random forest 
oob_accuracy_RF4 = Tree_RF_Best4.oob_score_


In [ ]:
#This function gives you the accuracy, precision, recall, f1, and cohen kappa score for both sets
def report_metrics(y_true, y_pred, label):
    print(label + "_Accuracy:", round(accuracy_score(y_true, y_pred), 3))
    print(label + "_Precision:", round(precision_score(y_true, y_pred), 3))
    print(label + "_Recall:", round(recall_score(y_true, y_pred), 3))
    print(label + "_F1_score:", round(f1_score(y_true, y_pred), 3))
    print(label + "_Kappa:", round(cohen_kappa_score(y_true, y_pred), 3))
    print()
#The performance metrics for train and validation are printed 
report_metrics(y_train3, y_train3_pred_RF, "RF_train")
report_metrics(y_val3, y_val3_pred_RF, "RF_validation")
report_metrics(y_test2, y_test2_pred_RF, "RF_test")
#The OOB accuracy is printed and compared to the validation  accuracy 
print("RF_OOB_Accuracy:", round(oob_accuracy_RF4, 3))
print("Validation vs OOB Accuracy Difference:", round(accuracy_score(y_val3, y_val3_pred_RF) - oob_accuracy_RF4, 3))
print("Test vs OOB Accuracy Difference:", round(accuracy_score(y_test2, y_test2_pred_RF) - oob_accuracy_RF4, 3))



In [ ]:
#The feature importance scores for each column is calculated with the best random forest model  
feature_importances = Tree_RF_Best4.feature_importances_

#A dicitonary with the column names and each columns feature importance score is made 
feature_importance_dict = {
    'feature_name': X_train3.columns,
    'feature_importances': feature_importances
}
#That dictionary is then converted into a dataframe and is printed in order from highest feature importance to lowest 
feature_importance_df = pd.DataFrame(feature_importance_dict).sort_values(
    by='feature_importances', ascending=False
)

print(feature_importance_df.round(3))


The model I chose is the pruned tree with the second data refinement method because it had the highest cohen kappa score which is a 0.461, the highest F1 score of 0.575, and the highest recall score of 0.536. While the random forest model with the second data refinement method had the best accuracy score of 0.829 and the best precision score of 0.675, the pruned tree model has the better performance because it will be able to find the most positive cases even if it won't be accurate. Additionally, I believe the Cohen kappa score is the most important to look at and the pruned tree has the highest out of all three.  
